[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto3/3pipeline.ipynb)
Confirmed runtime version: 2026.04

## Setup

In [ ]:
import sys

vi = sys.version_info
if not ((3, 12) <= (vi.major, vi.minor) < (3, 13)):
    raise RuntimeError(f"Python 3.12 required, got {vi.major}.{vi.minor}.{vi.micro}")

print(f"Python {vi.major}.{vi.minor}.{vi.micro} ✓")

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
print("Setup complete.")

## Data Models

In [ ]:
# Run `python proto3/sync_generated.py` after proto3/src/uol_fp/models.py
# changes to regenerate the block below -- do not edit it by hand.
# BEGIN AUTO-GENERATED (sync_generated.py)
from typing import Self

from pydantic import BaseModel, ConfigDict, Field, model_validator

ROLES = ["TechnicalMethod", "Task", "Dataset", "EvaluationMetric"]


class Evidence(BaseModel):
    model_config = ConfigDict(extra="forbid")

    section: str = Field(description="Exact section heading containing the quote.")
    quote: str = Field(
        description="One sentence quoted verbatim from the paper, supporting answer."
    )


class RoleExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    answer: str | None = Field(
        description="Shortest identifying term (e.g. 'Transformer'), or null if absent."
    )
    evidence: Evidence | None = Field(
        description="Evidence supporting the answer, or null when not present."
    )

    @model_validator(mode="after")
    def answer_and_evidence_must_match(self) -> Self:
        if (self.answer is None) != (self.evidence is None):
            raise ValueError(
                "answer and evidence must either both be null or both be present"
            )
        return self


class MethodologyProfile(BaseModel):
    model_config = ConfigDict(extra="forbid")

    TechnicalMethod: RoleExtraction
    Task: RoleExtraction
    Dataset: RoleExtraction
    EvaluationMetric: RoleExtraction


class RoleAnswer(BaseModel):
    model_config = ConfigDict(extra="forbid")

    answer: str = Field(description="Shortest identifying term for this valid answer.")
    evidence: Evidence = Field(description="Evidence supporting this specific answer.")


class MultiValuedRoleExtraction(BaseModel):
    """Pilot: a role that may legitimately have more than one valid answer at
    different granularities (e.g. a paper's broad self-description vs. the
    specific benchmark it evaluates on) -- see proto3/memo.md "Architecture
    reconsideration" result note (2026-08-29)."""

    model_config = ConfigDict(extra="forbid")

    answers: list[RoleAnswer] = Field(
        description=(
            "One or more valid answers for this role, ordered primary "
            "(most specific / directly evaluated) first. Empty list if the "
            "role is not present in the paper."
        )
    )


# END AUTO-GENERATED

print("Models ready.")

## Configuration — Select Paper

Set `PAPER_SLUG` below to the paper you are about to upload. Then use Colab's **Run focused cell and all cells below** starting from the next cell — upload, extraction, and evaluation all rerun with no further edits.

In [ ]:
# one of: transformer, bert, alexnet, resnet, mapreduce, pagerank
PAPER_SLUG = "transformer"
print(f"PAPER_SLUG = {PAPER_SLUG!r}")

## Stage 0 — Parse XML

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []

for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    h_lower = heading.lower().strip()

    if h_lower in SKIP_HEADINGS:
        continue

    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded: {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Stage 1 — Text Extraction

In [ ]:
document_text = ""
for s in sections:
    document_text += f"## {s['heading']}\n\n{s['text']}\n\n"

print(f"Document length: {len(document_text)} characters")
print(document_text[:500])

## Stage 2 — LLM Extraction (Gemini)

Before running this section, add your API key as a Colab secret: click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`, and enable notebook access.

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL_NAME = "gemini-3.5-flash"

print("Gemini client ready.")

## Stage 2b — Prompt Template

In [ ]:
PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "For each of the four roles below, identify the answer and its "
    "supporting evidence.\n"
    "\n"
    "Roles:\n"
    "- TechnicalMethod: the main method, model, algorithm, or system "
    "proposed by the authors\n"
    "- Task: the research task or problem being addressed\n"
    "- Dataset: the dataset used for training or evaluation\n"
    "- EvaluationMetric: the metric used to report results\n"
    "\n"
    "Rules:\n"
    "- Use the authors' own method, not methods cited from prior work.\n"
    "- Return null when a role is not present in the paper.\n"
    "- Evidence quotes must be copied verbatim from the paper, not "
    "paraphrased.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

prompt = PROMPT_TEMPLATE.format(paper_text=document_text)
print(f"Prompt length: {len(prompt)} characters")

## Stage 2c — Call Gemini and Parse Response

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=MethodologyProfile.model_json_schema(),
    ),
)

profile = MethodologyProfile.model_validate_json(response.text)

for role in ROLES:
    print(f"{role}: {getattr(profile, role)}")

print()
print(profile.model_dump_json(indent=2))

## Stage 2d — Decomposed Extraction (Variant B)

Variant B: independent Gemini calls, one per role, using a role-specific prompt instead of the joint prompt above — one prompt cell + one call cell per role. All four roles (`TechnicalMethod`, `Task`, `Dataset`, `EvaluationMetric`) are now implemented.

See `proto3/memo.md` "Architecture reconsideration" for the design rationale.

After running all four role cells for a paper, run the "Combine results" cell below once — it prints a single JSON block covering all four roles. Save that block by hand to `proto3/results_b/{PAPER_SLUG}.json`, one paste per paper, same manual-save convention as `proto3/results/run{N}/*.json`.

In [ ]:
TECHNICAL_METHOD_PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "Identify the TechnicalMethod: the main method, model, algorithm, or "
    "system proposed by the authors.\n"
    "\n"
    "Rules:\n"
    "- Use the authors' own method, not methods cited from prior work.\n"
    "- Distinguish the primary method from a component of it and from "
    "prior work -- do not return a component or a method the paper only "
    "cites.\n"
    "- Return null when the role is not present in the paper.\n"
    "- Evidence quotes must be copied verbatim from the paper, not "
    "paraphrased.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

technical_method_prompt = TECHNICAL_METHOD_PROMPT_TEMPLATE.format(
    paper_text=document_text
)
print(f"Prompt length: {len(technical_method_prompt)} characters")

In [ ]:
technical_method_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=technical_method_prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=RoleExtraction.model_json_schema(),
    ),
)

technical_method_extraction = RoleExtraction.model_validate_json(
    technical_method_response.text
)

print(f"TechnicalMethod: {technical_method_extraction}")
print()
print(technical_method_extraction.model_dump_json(indent=2))

In [ ]:
TASK_PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "Identify the Task: the research task or problem being addressed.\n"
    "\n"
    "Rules:\n"
    "- Identify the problem the paper actually solves, not a problem it "
    "only mentions as motivation or attributes to prior work.\n"
    "- Return null when the role is not present in the paper.\n"
    "- Evidence quotes must be copied verbatim from the paper, not "
    "paraphrased.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

task_prompt = TASK_PROMPT_TEMPLATE.format(paper_text=document_text)
print(f"Prompt length: {len(task_prompt)} characters")

In [ ]:
task_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=task_prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=RoleExtraction.model_json_schema(),
    ),
)

task_extraction = RoleExtraction.model_validate_json(task_response.text)

print(f"Task: {task_extraction}")
print()
print(task_extraction.model_dump_json(indent=2))

In [ ]:
DATASET_PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "Identify the Dataset: the dataset used for training or evaluation.\n"
    "\n"
    "Rules:\n"
    "- Identify the dataset actually used for training or evaluation, "
    "not a dataset only mentioned in passing or used by prior work.\n"
    "- Return null when the role is not present in the paper.\n"
    "- Evidence quotes must be copied verbatim from the paper, not "
    "paraphrased.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

dataset_prompt = DATASET_PROMPT_TEMPLATE.format(paper_text=document_text)
print(f"Prompt length: {len(dataset_prompt)} characters")

In [ ]:
dataset_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=dataset_prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=RoleExtraction.model_json_schema(),
    ),
)

dataset_extraction = RoleExtraction.model_validate_json(dataset_response.text)

print(f"Dataset: {dataset_extraction}")
print()
print(dataset_extraction.model_dump_json(indent=2))

In [ ]:
EVALUATION_METRIC_PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "Identify the EvaluationMetric: the metric used to report results.\n"
    "\n"
    "Rules:\n"
    "- Identify the metric actually used to report the paper's own "
    "results, not a metric only discussed or used by prior work.\n"
    "- Return null when the role is not present in the paper.\n"
    "- Evidence quotes must be copied verbatim from the paper, not "
    "paraphrased.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

evaluation_metric_prompt = EVALUATION_METRIC_PROMPT_TEMPLATE.format(
    paper_text=document_text
)
print(f"Prompt length: {len(evaluation_metric_prompt)} characters")

In [ ]:
evaluation_metric_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=evaluation_metric_prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=RoleExtraction.model_json_schema(),
    ),
)

evaluation_metric_extraction = RoleExtraction.model_validate_json(
    evaluation_metric_response.text
)

print(f"EvaluationMetric: {evaluation_metric_extraction}")
print()
print(evaluation_metric_extraction.model_dump_json(indent=2))

In [ ]:
# Combine results -- run once per paper, after every role cell above has run.
# Prints one JSON block covering every Variant B role implemented so far;
# paste it as-is into proto3/results_b/{PAPER_SLUG}.json.
import json

VARIANT_B_PROFILE = {
    "TechnicalMethod": technical_method_extraction,
    "Task": task_extraction,
    "Dataset": dataset_extraction,
    "EvaluationMetric": evaluation_metric_extraction,
}

print(
    json.dumps(
        {role: e.model_dump() for role, e in VARIANT_B_PROFILE.items()},
        indent=2,
    )
)

## Stage 3 — Evaluation

Score the pipeline output against gold labels and the frozen baseline (`proto3/baseline/*.json`), using a classification-style Precision/Recall/F1 metric instead of a substring-match count. See `proto3/memo.md` for the full 3-axis evaluation design; this implements axis 1 (gold label match).

In [ ]:
GOLD_LABELS = {
    "transformer": {
        "TechnicalMethod": "Transformer",
        "Task": "machine translation",
        "Dataset": "WMT",
        "EvaluationMetric": "BLEU",
    },
    "bert": {
        "TechnicalMethod": "BERT",
        "Task": "GLUE",
        "Dataset": "BooksCorpus",
        "EvaluationMetric": "F1",
    },
    "alexnet": {
        "TechnicalMethod": "convolutional",
        "Task": "object recognition",
        "Dataset": "ImageNet",
        "EvaluationMetric": "top-5",
    },
    "resnet": {
        "TechnicalMethod": "residual",
        "Task": "image recognition",
        "Dataset": "ImageNet",
        "EvaluationMetric": "top-1",
    },
    "mapreduce": {
        "TechnicalMethod": "MapReduce",
        "Task": "distributed",
        "Dataset": "TeraSort",
        "EvaluationMetric": "seconds",
    },
    "pagerank": {
        "TechnicalMethod": "PageRank",
        "Task": "web search",
        "Dataset": "million pages",
        "EvaluationMetric": "quality",
    },
}

print(f"Gold labels for {len(GOLD_LABELS)} papers loaded.")

In [ ]:
# Run `python proto3/sync_generated.py` after proto3/src/uol_fp/scoring.py
# changes to regenerate the block below -- do not edit it by hand.
# BEGIN AUTO-GENERATED (sync_generated.py)
ROLES = ["TechnicalMethod", "Task", "Dataset", "EvaluationMetric"]


def normalize(s: str | None) -> str | None:
    if s is None:
        return None
    return " ".join(s.lower().split())


def matches(gold: str, sys: str) -> bool:
    g, s = normalize(gold), normalize(sys)
    assert g is not None and s is not None
    return g in s or s in g


def score_role(gold: str | None, sys: str | None) -> tuple[int, int, int, int]:
    """Return (tp, fp, fn, tn) for one (paper, role) slot."""
    if gold is None:
        if sys is None:
            return (0, 0, 0, 1)
        return (0, 1, 0, 0)
    if sys is None:
        return (0, 0, 1, 0)
    if matches(gold, sys):
        return (1, 0, 0, 0)
    # both present but do not match: a confident wrong answer costs both
    # precision and recall
    return (0, 1, 1, 0)


def score_role_multi(
    gold: str | None, sys_answers: list[str]
) -> tuple[int, int, int, int]:
    """Return (tp, fp, fn, tn) for one (paper, role) slot, pilot multi-valued
    variant: sys_answers is a list of candidate answers (e.g. from
    MultiValuedRoleExtraction); a match against gold at any list position
    counts as a hit, same semantics as score_role otherwise."""
    if gold is None:
        if not sys_answers:
            return (0, 0, 0, 1)
        return (0, 1, 0, 0)
    if not sys_answers:
        return (0, 0, 1, 0)
    if any(matches(gold, sys) for sys in sys_answers):
        return (1, 0, 0, 0)
    # present but none match: a confident wrong answer costs both precision
    # and recall, same as score_role
    return (0, 1, 1, 0)


def score_profile(
    gold_answers: dict[str, str | None], sys_answers: dict[str, str | None]
) -> dict[str, tuple[int, int, int, int]]:
    return {
        role: score_role(gold_answers.get(role), sys_answers.get(role))
        for role in ROLES
    }


def precision_recall_f1(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


# END AUTO-GENERATED

print("Scoring functions ready.")

In [ ]:
# Run `python proto3/update_baseline.py` after proto3/baseline/*.json changes to
# regenerate the block below -- do not edit it by hand.
# BEGIN AUTO-GENERATED (update_baseline.py)
BASELINE_ANSWERS = {
    "alexnet": {
        "TechnicalMethod": "deep convolutional neural network",
        "Task": "object recognition",
        "Dataset": "ImageNet",
        "EvaluationMetric": "top-1 and top-5 error rates",
    },
    "bert": {
        "TechnicalMethod": "BERT",
        "Task": "Language model pre-training",
        "Dataset": "SQuAD v1.1",
        "EvaluationMetric": "F1 score",
    },
    "mapreduce": {
        "TechnicalMethod": "MapReduce",
        "Task": (
            "automatic parallelization and distribution of large-scale computations"
        ),
        "Dataset": None,
        "EvaluationMetric": "elapsed time",
    },
    "pagerank": {
        "TechnicalMethod": "Google",
        "Task": "information retrieval",
        "Dataset": "24 million pages",
        "EvaluationMetric": None,
    },
    "resnet": {
        "TechnicalMethod": "deep residual learning framework",
        "Task": "image classification",
        "Dataset": "ImageNet 2012 classification dataset",
        "EvaluationMetric": "top-1 and top-5 error rates",
    },
    "transformer": {
        "TechnicalMethod": "Transformer",
        "Task": "machine translation",
        "Dataset": "WMT 2014 English-German",
        "EvaluationMetric": "BLEU",
    },
}
# END AUTO-GENERATED


def aggregate_scores(
    results: dict[str, dict[str, str | None]],
) -> dict[str, tuple[int, int, int, int]]:
    totals = {role: (0, 0, 0, 0) for role in ROLES}
    for slug, sys_answers in results.items():
        per_role = score_profile(GOLD_LABELS[slug], sys_answers)
        for role, (tp, fp, fn, tn) in per_role.items():
            t_tp, t_fp, t_fn, t_tn = totals[role]
            totals[role] = (t_tp + tp, t_fp + fp, t_fn + fn, t_tn + tn)
    return totals


def print_score_table(totals: dict[str, tuple[int, int, int, int]], title: str) -> None:
    print(title)
    print(
        f"{'Role':<18} {'TP':>3} {'FP':>3} {'FN':>3} {'TN':>3}"
        f"  {'P':>5} {'R':>5} {'F1':>5}"
    )
    overall = (0, 0, 0, 0)
    for role in ROLES:
        tp, fp, fn, tn = totals[role]
        p, r, f1 = precision_recall_f1(tp, fp, fn)
        print(
            f"{role:<18} {tp:>3} {fp:>3} {fn:>3} {tn:>3}"
            f"  {p:>5.2f} {r:>5.2f} {f1:>5.2f}"
        )
        overall = tuple(a + b for a, b in zip(overall, (tp, fp, fn, tn)))
    p, r, f1 = precision_recall_f1(overall[0], overall[1], overall[2])
    print(
        f"{'Overall':<18} {overall[0]:>3} {overall[1]:>3} {overall[2]:>3}"
        f" {overall[3]:>3}"
        f"  {p:>5.2f} {r:>5.2f} {f1:>5.2f}"
    )
    print()


BASELINE_TOTALS = aggregate_scores(BASELINE_ANSWERS)
print_score_table(BASELINE_TOTALS, "Baseline (frozen proto3/baseline/*.json) vs gold:")

In [ ]:
pipeline_answers = {role: getattr(profile, role).answer for role in ROLES}
gold_answers = GOLD_LABELS[PAPER_SLUG]
baseline_answers = BASELINE_ANSWERS[PAPER_SLUG]

print(f"{'Role':<18} {'Gold':<25} {'Baseline':<25} {'Pipeline':<25}")
for role in ROLES:
    gold = gold_answers.get(role) or "(none)"
    baseline = baseline_answers.get(role) or "(none)"
    pipeline = pipeline_answers.get(role) or "(none)"
    print(f"{role:<18} {gold:<25} {baseline:<25} {pipeline:<25}")
print()

print_score_table(
    score_profile(gold_answers, baseline_answers), f"Baseline vs gold ({PAPER_SLUG}):"
)
print_score_table(
    score_profile(gold_answers, pipeline_answers), f"Pipeline vs gold ({PAPER_SLUG}):"
)

In [ ]:
# Variant B TechnicalMethod vs gold, alongside Variant A (joint pipeline) from
# the table above. See Stage 2d.
variant_b_technical_method = technical_method_extraction.answer
gold_technical_method = gold_answers["TechnicalMethod"]

print(f"{'Role':<18} {'Gold':<25} {'Pipeline (A)':<25} {'Variant B':<25}")
print(
    f"{'TechnicalMethod':<18} {gold_technical_method:<25} "
    f"{(pipeline_answers.get('TechnicalMethod') or '(none)'):<25} "
    f"{(variant_b_technical_method or '(none)'):<25}"
)
print()

tp, fp, fn, tn = score_role(gold_technical_method, variant_b_technical_method)
p, r, f1 = precision_recall_f1(tp, fp, fn)
print(
    f"Variant B TechnicalMethod vs gold ({PAPER_SLUG}): "
    f"P={p:.2f} R={r:.2f} F1={f1:.2f}"
)

In [ ]:
# Variant B Task vs gold, alongside Variant A (joint pipeline) from the table
# above. See Stage 2d.
variant_b_task = task_extraction.answer
gold_task = gold_answers["Task"]

print(f"{'Role':<18} {'Gold':<25} {'Pipeline (A)':<25} {'Variant B':<25}")
print(
    f"{'Task':<18} {gold_task:<25} "
    f"{(pipeline_answers.get('Task') or '(none)'):<25} "
    f"{(variant_b_task or '(none)'):<25}"
)
print()

tp, fp, fn, tn = score_role(gold_task, variant_b_task)
p, r, f1 = precision_recall_f1(tp, fp, fn)
print(f"Variant B Task vs gold ({PAPER_SLUG}): P={p:.2f} R={r:.2f} F1={f1:.2f}")

In [ ]:
# Variant B Dataset vs gold, alongside Variant A (joint pipeline) from the table
# above. See Stage 2d.
variant_b_dataset = dataset_extraction.answer
gold_dataset = gold_answers["Dataset"]

print(f"{'Role':<18} {'Gold':<25} {'Pipeline (A)':<25} {'Variant B':<25}")
print(
    f"{'Dataset':<18} {gold_dataset:<25} "
    f"{(pipeline_answers.get('Dataset') or '(none)'):<25} "
    f"{(variant_b_dataset or '(none)'):<25}"
)
print()

tp, fp, fn, tn = score_role(gold_dataset, variant_b_dataset)
p, r, f1 = precision_recall_f1(tp, fp, fn)
print(f"Variant B Dataset vs gold ({PAPER_SLUG}): P={p:.2f} R={r:.2f} F1={f1:.2f}")

In [ ]:
# Variant B EvaluationMetric vs gold, alongside Variant A (joint pipeline) from
# the table above. See Stage 2d.
variant_b_evaluation_metric = evaluation_metric_extraction.answer
gold_evaluation_metric = gold_answers["EvaluationMetric"]

print(f"{'Role':<18} {'Gold':<25} {'Pipeline (A)':<25} {'Variant B':<25}")
print(
    f"{'EvaluationMetric':<18} {gold_evaluation_metric:<25} "
    f"{(pipeline_answers.get('EvaluationMetric') or '(none)'):<25} "
    f"{(variant_b_evaluation_metric or '(none)'):<25}"
)
print()

tp, fp, fn, tn = score_role(gold_evaluation_metric, variant_b_evaluation_metric)
p, r, f1 = precision_recall_f1(tp, fp, fn)
print(
    f"Variant B EvaluationMetric vs gold ({PAPER_SLUG}): "
    f"P={p:.2f} R={r:.2f} F1={f1:.2f}"
)

## Stage 2e — Multi-valued Task Pilot

Pilot, not part of Variant B/C: tests whether Task's F1 weakness is partly a single-valued-schema artifact rather than only a model/prompt problem. Motivated by a concrete case found while comparing Variant A/B (`proto3/memo.md` "Architecture reconsideration" result note, 2026-08-29): Transformer's Task plausibly has two valid answers at different granularities ("sequence transduction" — the broad architecture class the paper claims — vs "machine translation" — the specific benchmark it is evaluated on), and a single-valued prompt/schema forces a choice between them regardless of which one matches gold.

Uses `MultiValuedRoleExtraction` (`proto3/src/uol_fp/models.py`) — a list of `RoleAnswer`, ordered most specific first — and `score_role_multi` (`proto3/src/uol_fp/scoring.py`), which counts a match against gold at *any* list position as a hit. Small, Task-only pilot: no other role is changed.

In [ ]:
MULTI_VALUED_TASK_PROMPT_TEMPLATE = (
    "You are extracting research methodology from a computing research "
    "paper.\n"
    "\n"
    "Identify every valid description of the Task: the research task or "
    "problem this paper addresses, at every granularity that is directly "
    "supported by the paper's own text -- from the specific benchmark or "
    "experiment actually evaluated, to the broader problem class the "
    "authors describe their own contribution as belonging to.\n"
    "\n"
    "Rules:\n"
    "- List one entry per distinct valid description, most specific "
    "first (the task actually evaluated), most general last (the "
    "authors' own broader self-description).\n"
    "- Only include descriptions the paper makes as its own claim, not "
    "ones it only mentions as motivation or attributes to prior work.\n"
    "- Each entry needs its own evidence quote, copied verbatim.\n"
    "- Return an empty list when the role is not present in the paper.\n"
    "\n"
    "Paper text:\n"
    "{paper_text}\n"
)

multi_valued_task_prompt = MULTI_VALUED_TASK_PROMPT_TEMPLATE.format(
    paper_text=document_text
)
print(f"Prompt length: {len(multi_valued_task_prompt)} characters")

In [ ]:
multi_valued_task_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=multi_valued_task_prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        seed=0,
        response_mime_type="application/json",
        response_json_schema=MultiValuedRoleExtraction.model_json_schema(),
    ),
)

multi_valued_task_extraction = MultiValuedRoleExtraction.model_validate_json(
    multi_valued_task_response.text
)

print(f"Task (multi-valued): {multi_valued_task_extraction}")
print()
print(multi_valued_task_extraction.model_dump_json(indent=2))

In [ ]:
# Multi-valued Task pilot vs gold, alongside Variant A/B single-valued Task
# from the cells above. See Stage 2e.
multi_valued_task_answers = [a.answer for a in multi_valued_task_extraction.answers]
gold_task_multi = gold_answers["Task"]

print(
    f"{'Role':<18} {'Gold':<25} {'Pipeline (A)':<25} {'Variant B':<25}"
    f" {'Multi-valued'}"
)
print(
    f"{'Task':<18} {gold_task_multi:<25} "
    f"{(pipeline_answers.get('Task') or '(none)'):<25} "
    f"{(variant_b_task or '(none)'):<25} "
    f"{', '.join(multi_valued_task_answers) or '(none)'}"
)
print()

tp, fp, fn, tn = score_role_multi(gold_task_multi, multi_valued_task_answers)
p, r, f1 = precision_recall_f1(tp, fp, fn)
print(
    f"Multi-valued Task vs gold ({PAPER_SLUG}): P={p:.2f} R={r:.2f} F1={f1:.2f}"
)

In [ ]:
# Accumulates across multiple runs of Stage 0 -> 2c -> 3 (per-paper cell above) within
# one kernel session -- one entry per paper, without restarting the runtime.
if "PIPELINE_RESULTS" not in globals():
    PIPELINE_RESULTS: dict[str, dict[str, str | None]] = {}

PIPELINE_RESULTS[PAPER_SLUG] = pipeline_answers
print(f"Recorded: {sorted(PIPELINE_RESULTS)}")

if len(PIPELINE_RESULTS) == len(GOLD_LABELS):
    pipeline_totals = aggregate_scores(PIPELINE_RESULTS)
    print()
    print_score_table(BASELINE_TOTALS, "Baseline vs gold (all 6 papers):")
    print_score_table(pipeline_totals, "Pipeline vs gold (all 6 papers):")
else:
    missing = sorted(set(GOLD_LABELS) - set(PIPELINE_RESULTS))
    print(f"Waiting for {len(missing)} more paper(s): {missing}")